# Compare Original ROBERT and ChatBob-Modified ROBERT Outputs

## Purpose

This notebook compares the outputs from two completed ROBERT runs:

1. **Original ROBERT**, an unchanged reference version of ROBERT.
2. **ChatBob ROBERT**, the modified version that adds structured JSON outputs for later use by the ChatBob interface and companion agent.

The purpose of this comparison is to confirm that the ChatBob-related modifications do not change ROBERT's standard scientific outputs.

ROBERT remains the scientific source of truth. The JSON layer is intended to record evidence that ROBERT already produces without changing its:

- data-curation behavior,
- descriptor selection,
- model generation and model selection,
- verification tests,
- predictions,
- scoring,
- standard `.dat` files,
- standard `.csv` files,
- plots, or
- PDF report.

## Assumptions

This notebook does **not** run ROBERT itself.

Before using the notebook, the user must run the same dataset through both versions of ROBERT and place the completed outputs into two separate folders.

The expected comparison structure is:

```text
comparison/
├── original_robert/
│   ├── CURATE/
│   ├── GENERATE/
│   ├── VERIFY/
│   ├── PREDICT/
│   └── ROBERT_report.pdf
│
├── chatbob_robert/
│   ├── CURATE/
│   ├── GENERATE/
│   ├── VERIFY/
│   ├── PREDICT/
│   ├── ROBERT_report.pdf
│   └── additional JSON files
│
└── compare_robert_outputs.ipynb

In [1]:
from pathlib import Path

# The notebook should be stored inside the comparison folder:
#
# comparison/
# ├── compare_robert_outputs.ipynb
# ├── original_robert/
# └── chatbob_robert/
#
# These paths are resolved relative to the directory from which the notebook
# kernel is running, which should normally be the comparison folder.

COMPARISON_ROOT = Path.cwd().resolve()

ORIGINAL_RUN = COMPARISON_ROOT / "original_robert"
CHATBOB_RUN = COMPARISON_ROOT / "chatbob_robert"

print(f"Comparison folder:    {COMPARISON_ROOT}")
print(f"Original ROBERT run:  {ORIGINAL_RUN}")
print(f"ChatBob ROBERT run:   {CHATBOB_RUN}")

Comparison folder:    /Users/cjcscha/ROBERT/chat_rob_UI/robert/comparison
Original ROBERT run:  /Users/cjcscha/ROBERT/chat_rob_UI/robert/comparison/original_robert
ChatBob ROBERT run:   /Users/cjcscha/ROBERT/chat_rob_UI/robert/comparison/chatbob_robert


In [2]:
# Validate the comparison folder structure before attempting any file comparisons.

expected_run_folders = {
    "Original ROBERT": ORIGINAL_RUN,
    "ChatBob ROBERT": CHATBOB_RUN,
}

print(f"Notebook working directory: {COMPARISON_ROOT}\n")

all_paths_valid = True

for label, run_path in expected_run_folders.items():
    if run_path.exists() and run_path.is_dir():
        print(f"PASS: {label} folder found")
        print(f"      {run_path}")
    else:
        print(f"FAIL: {label} folder not found")
        print(f"      Expected location: {run_path}")
        all_paths_valid = False

if all_paths_valid:
    print("\nPASS: Both ROBERT run folders are available.")
else:
    print(
        "\nACTION REQUIRED: Check the notebook working directory and the "
        "folder names defined in Cell 2."
    )

Notebook working directory: /Users/cjcscha/ROBERT/chat_rob_UI/robert/comparison

PASS: Original ROBERT folder found
      /Users/cjcscha/ROBERT/chat_rob_UI/robert/comparison/original_robert
PASS: ChatBob ROBERT folder found
      /Users/cjcscha/ROBERT/chat_rob_UI/robert/comparison/chatbob_robert

PASS: Both ROBERT run folders are available.


In [3]:
# Check that both run folders contain the expected standard ROBERT outputs.
#
# A missing item does not automatically mean the run failed. For example,
# ROBERT_report.pdf may be absent if PDF-generation dependencies were unavailable.
# This cell records what is present before we begin comparing file contents.

EXPECTED_DIRECTORIES = [
    "CURATE",
    "GENERATE",
    "VERIFY",
    "PREDICT",
]

EXPECTED_ROOT_FILES = [
    "ROBERT_report.pdf",
]

run_structure_results = {}

for run_label, run_path in expected_run_folders.items():
    print(f"\n{run_label}")
    print("=" * len(run_label))

    directory_results = {}
    file_results = {}

    for directory_name in EXPECTED_DIRECTORIES:
        directory_path = run_path / directory_name
        exists = directory_path.exists() and directory_path.is_dir()
        directory_results[directory_name] = exists

        status = "PASS" if exists else "MISSING"
        print(f"{status:7} Directory: {directory_name}/")

    for file_name in EXPECTED_ROOT_FILES:
        file_path = run_path / file_name
        exists = file_path.exists() and file_path.is_file()
        file_results[file_name] = exists

        status = "PASS" if exists else "MISSING"
        print(f"{status:7} File:      {file_name}")

    run_structure_results[run_label] = {
        "directories": directory_results,
        "files": file_results,
    }


Original ROBERT
PASS    Directory: CURATE/
PASS    Directory: GENERATE/
PASS    Directory: VERIFY/
PASS    Directory: PREDICT/
PASS    File:      ROBERT_report.pdf

ChatBob ROBERT
PASS    Directory: CURATE/
PASS    Directory: GENERATE/
PASS    Directory: VERIFY/
PASS    Directory: PREDICT/
PASS    File:      ROBERT_report.pdf


In [4]:
# Collect all standard files from both ROBERT output folders.
# The notebook assumes the user has already placed the correct files
# into original_robert/ and chatbob_robert/.

def collect_relative_files(run_path):
    return {
        path.relative_to(run_path)
        for path in run_path.rglob("*")
        if path.is_file()
    }

original_files = collect_relative_files(ORIGINAL_RUN)
chatbob_files = collect_relative_files(CHATBOB_RUN)

common_files = sorted(original_files & chatbob_files)
original_only_files = sorted(original_files - chatbob_files)
chatbob_only_files = sorted(chatbob_files - original_files)

print(f"Matching file paths: {len(common_files)}")
print(f"Only in original:    {len(original_only_files)}")
print(f"Only in ChatBob:     {len(chatbob_only_files)}")

Matching file paths: 53
Only in original:    3
Only in ChatBob:     0


In [5]:
if original_only_files:
    print("\nOnly in original ROBERT:")
    for path in original_only_files:
        print(path)

if chatbob_only_files:
    print("\nOnly in ChatBob ROBERT:")
    for path in chatbob_only_files:
        print(path)


Only in original ROBERT:
GENERATE/.DS_Store
GENERATE/Best_model/.DS_Store
GENERATE/Raw_data/.DS_Store


In [6]:
from pathlib import Path
import difflib

# Compare matching ROBERT .dat files.
# The first check is exact: are the two files byte-for-byte identical?
# If not, show a short unified diff so the differences can be inspected.

dat_files = sorted(
    path for path in common_files
    if path.suffix.lower() == ".dat"
)

print(f"Matching .dat files found: {len(dat_files)}\n")

dat_comparison_results = {}

for relative_path in dat_files:
    original_path = ORIGINAL_RUN / relative_path
    chatbob_path = CHATBOB_RUN / relative_path

    original_bytes = original_path.read_bytes()
    chatbob_bytes = chatbob_path.read_bytes()

    identical = original_bytes == chatbob_bytes
    dat_comparison_results[str(relative_path)] = identical

    status = "IDENTICAL" if identical else "DIFFERENT"
    print(f"{status:10} {relative_path}")

    if not identical:
        original_lines = original_path.read_text(
            errors="replace"
        ).splitlines()

        chatbob_lines = chatbob_path.read_text(
            errors="replace"
        ).splitlines()

        diff = list(
            difflib.unified_diff(
                original_lines,
                chatbob_lines,
                fromfile=f"original/{relative_path}",
                tofile=f"chatbob/{relative_path}",
                lineterm="",
            )
        )

        print("\n".join(diff[:40]))

        if len(diff) > 40:
            print(f"... {len(diff) - 40} additional diff lines not shown")

        print()

Matching .dat files found: 4

DIFFERENT  CURATE/CURATE_data.dat
--- original/CURATE/CURATE_data.dat
+++ chatbob/CURATE/CURATE_data.dat
@@ -1,7 +1,7 @@
-ROBERT v 2.1.2 2026/07/10 14:21:21 
+ROBERT v 2.1.2 2026/08/31 12:37:04 
 How to cite: Dalmau, D.; Alegre Requena, J. V. WIREs Comput Mol Sci. 2024, 14, e1733.
 
-Command line used in ROBERT: python -m robert --csv_name "/Users/cjcscha/ROBERT/robert-1/H_predict_ln-k.csv" --y "ln(k)_rate" --names "Coupling" --ignore "Coupling"
+Command line used in ROBERT: python -m robert --csv_name "H_predict_ln-k.csv" --y "ln(k)_rate" --names "Coupling" --ignore "Coupling"
 
 
 o  Starting data curation with the CURATE module
@@ -38,9 +38,9 @@
    o Model MVL: 4 descriptors remaining:
       IR-freq-C=O_interm_boltz, NBO-C1 C=O_anion_boltz, Sterimol-B1-N1-C4_amine_boltz, Vbur-2.0A_anion_boltz
 
-o  Model-specific curated databases were stored in /Users/cjcscha/ROBERT/robert-1/CURATE
+o  Model-specific curated databases were stored in /Users/cjcscha/RO

In [7]:
import pandas as pd
import numpy as np

# Compare matching CSV files produced by the two ROBERT runs.
#
# Numeric values are compared using a small tolerance because values that are
# scientifically equivalent may sometimes differ only through floating-point
# formatting. Text values are compared exactly.
#
# This comparison assumes that corresponding rows appear in the same order.

RTOL = 1e-8
ATOL = 1e-10

csv_files = sorted(
    path for path in common_files
    if path.suffix.lower() == ".csv"
)

print(f"Matching CSV files found: {len(csv_files)}\n")

csv_comparison_results = {}

for relative_path in csv_files:
    original_path = ORIGINAL_RUN / relative_path
    chatbob_path = CHATBOB_RUN / relative_path

    original_df = pd.read_csv(original_path)
    chatbob_df = pd.read_csv(chatbob_path)

    result = {
        "same_shape": original_df.shape == chatbob_df.shape,
        "same_columns": list(original_df.columns) == list(chatbob_df.columns),
        "numeric_differences": None,
        "text_differences": None,
        "maximum_absolute_difference": None,
    }

    print(relative_path)
    print("-" * len(str(relative_path)))

    if not result["same_shape"]:
        print(
            f"DIFFERENT SHAPE: original {original_df.shape}, "
            f"ChatBob {chatbob_df.shape}\n"
        )
        csv_comparison_results[str(relative_path)] = result
        continue

    if not result["same_columns"]:
        print("DIFFERENT COLUMNS")
        print(f"Original: {list(original_df.columns)}")
        print(f"ChatBob:  {list(chatbob_df.columns)}\n")
        csv_comparison_results[str(relative_path)] = result
        continue

    numeric_columns = original_df.select_dtypes(
        include=[np.number]
    ).columns.tolist()

    text_columns = [
        column
        for column in original_df.columns
        if column not in numeric_columns
    ]

    numeric_difference_count = 0
    maximum_absolute_difference = 0.0

    for column in numeric_columns:
        original_values = original_df[column].to_numpy()
        chatbob_values = chatbob_df[column].to_numpy()

        values_match = np.isclose(
            original_values,
            chatbob_values,
            rtol=RTOL,
            atol=ATOL,
            equal_nan=True,
        )

        numeric_difference_count += int((~values_match).sum())

        finite_differences = np.abs(
            original_values.astype(float)
            - chatbob_values.astype(float)
        )

        finite_differences = finite_differences[
            np.isfinite(finite_differences)
        ]

        if finite_differences.size:
            maximum_absolute_difference = max(
                maximum_absolute_difference,
                float(finite_differences.max()),
            )

    text_difference_count = 0

    for column in text_columns:
        original_values = original_df[column].fillna("<MISSING>").astype(str)
        chatbob_values = chatbob_df[column].fillna("<MISSING>").astype(str)

        text_difference_count += int(
            (original_values != chatbob_values).sum()
        )

    result["numeric_differences"] = numeric_difference_count
    result["text_differences"] = text_difference_count
    result["maximum_absolute_difference"] = maximum_absolute_difference

    if numeric_difference_count == 0 and text_difference_count == 0:
        print("IDENTICAL within numerical tolerance")
    else:
        print(f"Numeric cells different: {numeric_difference_count}")
        print(f"Text cells different:    {text_difference_count}")
        print(
            "Maximum absolute numeric difference: "
            f"{maximum_absolute_difference:.12g}"
        )

    print()

    csv_comparison_results[str(relative_path)] = result

Matching CSV files found: 28

CURATE/CURATE_options.csv
-------------------------
Numeric cells different: 0
Text cells different:    1
Maximum absolute numeric difference: 0

CURATE/H_predict_ln-k_CURATE.csv
--------------------------------
IDENTICAL within numerical tolerance

CURATE/H_predict_ln-k_CURATE_GB.csv
-----------------------------------
IDENTICAL within numerical tolerance

CURATE/H_predict_ln-k_CURATE_MVL.csv
------------------------------------
IDENTICAL within numerical tolerance

CURATE/H_predict_ln-k_CURATE_NN.csv
-----------------------------------
IDENTICAL within numerical tolerance

CURATE/H_predict_ln-k_CURATE_RF.csv
-----------------------------------
IDENTICAL within numerical tolerance

GENERATE/Best_model/No_PFI/NN.csv
---------------------------------
IDENTICAL within numerical tolerance

GENERATE/Best_model/No_PFI/NN_db.csv
------------------------------------
IDENTICAL within numerical tolerance

GENERATE/Best_model/PFI/MVL_PFI.csv
------------------------

In [8]:
# Inspect CSV files that have different shapes or column structures.

for relative_path in csv_files:
    original_path = ORIGINAL_RUN / relative_path
    chatbob_path = CHATBOB_RUN / relative_path

    original_df = pd.read_csv(original_path)
    chatbob_df = pd.read_csv(chatbob_path)

    if (
        original_df.shape != chatbob_df.shape
        or list(original_df.columns) != list(chatbob_df.columns)
    ):
        original_columns = list(original_df.columns)
        chatbob_columns = list(chatbob_df.columns)

        only_in_original = [
            column for column in original_columns
            if column not in chatbob_columns
        ]

        only_in_chatbob = [
            column for column in chatbob_columns
            if column not in original_columns
        ]

        print(f"\n{relative_path}")
        print("=" * len(str(relative_path)))

        print(f"Original shape: {original_df.shape}")
        print(f"ChatBob shape:  {chatbob_df.shape}")

        print("\nOriginal columns:")
        for column in original_columns:
            print(f"  {column}")

        print("\nChatBob columns:")
        for column in chatbob_columns:
            print(f"  {column}")

        print("\nOnly in original:")
        if only_in_original:
            for column in only_in_original:
                print(f"  {column}")
        else:
            print("  None")

        print("\nOnly in ChatBob:")
        if only_in_chatbob:
            for column in only_in_chatbob:
                print(f"  {column}")
        else:
            print("  None")

In [9]:
# Compare only the columns shared by both versions of each CSV.
#
# This allows us to ignore additional columns introduced in one ROBERT version,
# such as ln(k)_rate_pred_conformal_hw, while still checking whether the common
# scientific outputs are identical.

shared_column_results = {}

for relative_path in csv_files:
    original_path = ORIGINAL_RUN / relative_path
    chatbob_path = CHATBOB_RUN / relative_path

    original_df = pd.read_csv(original_path)
    chatbob_df = pd.read_csv(chatbob_path)

    shared_columns = [
        column
        for column in original_df.columns
        if column in chatbob_df.columns
    ]

    original_shared = original_df[shared_columns]
    chatbob_shared = chatbob_df[shared_columns]

    if original_shared.shape != chatbob_shared.shape:
        print(f"{relative_path}: DIFFERENT ROW COUNTS")
        continue

    numeric_columns = original_shared.select_dtypes(
        include=[np.number]
    ).columns.tolist()

    text_columns = [
        column
        for column in shared_columns
        if column not in numeric_columns
    ]

    differing_numeric_cells = 0
    differing_text_cells = 0
    maximum_absolute_difference = 0.0

    for column in numeric_columns:
        original_values = original_shared[column].to_numpy(dtype=float)
        chatbob_values = chatbob_shared[column].to_numpy(dtype=float)

        matches = np.isclose(
            original_values,
            chatbob_values,
            rtol=RTOL,
            atol=ATOL,
            equal_nan=True,
        )

        differing_numeric_cells += int((~matches).sum())

        differences = np.abs(original_values - chatbob_values)
        finite_differences = differences[np.isfinite(differences)]

        if finite_differences.size:
            maximum_absolute_difference = max(
                maximum_absolute_difference,
                float(finite_differences.max()),
            )

    for column in text_columns:
        original_values = (
            original_shared[column]
            .fillna("<MISSING>")
            .astype(str)
        )

        chatbob_values = (
            chatbob_shared[column]
            .fillna("<MISSING>")
            .astype(str)
        )

        differing_text_cells += int(
            (original_values != chatbob_values).sum()
        )

    shared_column_results[str(relative_path)] = {
        "shared_columns": shared_columns,
        "numeric_differences": differing_numeric_cells,
        "text_differences": differing_text_cells,
        "maximum_absolute_difference": maximum_absolute_difference,
    }

    if differing_numeric_cells == 0 and differing_text_cells == 0:
        print(f"IDENTICAL shared columns: {relative_path}")
    else:
        print(f"\nDIFFERENT shared columns: {relative_path}")
        print(f"  Numeric cells different: {differing_numeric_cells}")
        print(f"  Text cells different:    {differing_text_cells}")
        print(
            "  Maximum absolute difference: "
            f"{maximum_absolute_difference:.12g}"
        )


DIFFERENT shared columns: CURATE/CURATE_options.csv
  Numeric cells different: 0
  Text cells different:    1
  Maximum absolute difference: 0
IDENTICAL shared columns: CURATE/H_predict_ln-k_CURATE.csv
IDENTICAL shared columns: CURATE/H_predict_ln-k_CURATE_GB.csv
IDENTICAL shared columns: CURATE/H_predict_ln-k_CURATE_MVL.csv
IDENTICAL shared columns: CURATE/H_predict_ln-k_CURATE_NN.csv
IDENTICAL shared columns: CURATE/H_predict_ln-k_CURATE_RF.csv
IDENTICAL shared columns: GENERATE/Best_model/No_PFI/NN.csv
IDENTICAL shared columns: GENERATE/Best_model/No_PFI/NN_db.csv
IDENTICAL shared columns: GENERATE/Best_model/PFI/MVL_PFI.csv
IDENTICAL shared columns: GENERATE/Best_model/PFI/MVL_PFI_db.csv
IDENTICAL shared columns: GENERATE/Raw_data/No_PFI/GB.csv
IDENTICAL shared columns: GENERATE/Raw_data/No_PFI/GB_db.csv
IDENTICAL shared columns: GENERATE/Raw_data/No_PFI/MVL.csv
IDENTICAL shared columns: GENERATE/Raw_data/No_PFI/MVL_db.csv
IDENTICAL shared columns: GENERATE/Raw_data/No_PFI/NN.csv


In [10]:
# Identify which shared columns differ in each CSV.
#
# For numeric columns, report:
# - number of differing rows,
# - maximum absolute difference,
# - mean absolute difference.
#
# For text columns, report the number of differing rows.

for relative_path in csv_files:
    original_path = ORIGINAL_RUN / relative_path
    chatbob_path = CHATBOB_RUN / relative_path

    original_df = pd.read_csv(original_path)
    chatbob_df = pd.read_csv(chatbob_path)

    shared_columns = [
        column
        for column in original_df.columns
        if column in chatbob_df.columns
    ]

    column_differences = []

    for column in shared_columns:
        original_column = original_df[column]
        chatbob_column = chatbob_df[column]

        if (
            pd.api.types.is_numeric_dtype(original_column)
            and pd.api.types.is_numeric_dtype(chatbob_column)
        ):
            original_values = original_column.to_numpy(dtype=float)
            chatbob_values = chatbob_column.to_numpy(dtype=float)

            matches = np.isclose(
                original_values,
                chatbob_values,
                rtol=RTOL,
                atol=ATOL,
                equal_nan=True,
            )

            differing_rows = int((~matches).sum())

            if differing_rows > 0:
                absolute_differences = np.abs(
                    original_values - chatbob_values
                )

                finite_differences = absolute_differences[
                    np.isfinite(absolute_differences)
                ]

                column_differences.append(
                    {
                        "column": column,
                        "type": "numeric",
                        "differing_rows": differing_rows,
                        "maximum_absolute_difference": (
                            float(finite_differences.max())
                            if finite_differences.size
                            else np.nan
                        ),
                        "mean_absolute_difference": (
                            float(finite_differences.mean())
                            if finite_differences.size
                            else np.nan
                        ),
                    }
                )

        else:
            original_values = (
                original_column.fillna("<MISSING>").astype(str)
            )
            chatbob_values = (
                chatbob_column.fillna("<MISSING>").astype(str)
            )

            differing_rows = int(
                (original_values != chatbob_values).sum()
            )

            if differing_rows > 0:
                column_differences.append(
                    {
                        "column": column,
                        "type": "text",
                        "differing_rows": differing_rows,
                    }
                )

    if column_differences:
        print(f"\n{relative_path}")
        print("=" * len(str(relative_path)))

        for result in column_differences:
            print(f"Column: {result['column']}")
            print(f"  Type:             {result['type']}")
            print(f"  Differing rows:   {result['differing_rows']}")

            if result["type"] == "numeric":
                print(
                    "  Maximum difference: "
                    f"{result['maximum_absolute_difference']:.12g}"
                )
                print(
                    "  Mean difference:    "
                    f"{result['mean_absolute_difference']:.12g}"
                )


CURATE/CURATE_options.csv
Column: csv_name
  Type:             text
  Differing rows:   1


In [11]:
import re

# Compare DAT files after removing expected run-specific differences.
#
# The normalization removes:
# - the run timestamp from the ROBERT version line,
# - the command-line CSV path,
# - absolute output-directory paths,
# - module execution times,
# - trailing whitespace.
#
# It does not remove scientific results.

def normalize_dat_text(path):
    lines = path.read_text(errors="replace").splitlines()
    normalized_lines = []

    for line in lines:
        # Remove trailing spaces and tabs.
        line = line.rstrip()

        # Normalize the timestamp while preserving the ROBERT version.
        line = re.sub(
            r"^(ROBERT v\s+\S+)\s+\d{4}/\d{2}/\d{2}\s+\d{2}:\d{2}:\d{2}\s*$",
            r"\1 <RUN_TIMESTAMP>",
            line,
        )

        # Normalize the input CSV path in the recorded command line.
        line = re.sub(
            r'(--csv_name\s+)"[^"]+"',
            r'\1"<INPUT_CSV>"',
            line,
        )

        # Normalize absolute paths used in output-location messages.
        if "were stored in " in line:
            line = re.sub(
                r"were stored in .*$",
                "were stored in <OUTPUT_DIRECTORY>",
                line,
            )

        # Normalize module execution times.
        line = re.sub(
            r"^Time (CURATE|GENERATE|VERIFY|PREDICT):\s+"
            r"[0-9.]+\s+seconds$",
            r"Time \1: <EXECUTION_TIME>",
            line,
        )

        normalized_lines.append(line)

    return "\n".join(normalized_lines)


print("NORMALIZED DAT COMPARISON")
print("=========================")

normalized_dat_results = {}

for relative_path in dat_files:
    original_path = ORIGINAL_RUN / relative_path
    chatbob_path = CHATBOB_RUN / relative_path

    original_normalized = normalize_dat_text(original_path)
    chatbob_normalized = normalize_dat_text(chatbob_path)

    identical = original_normalized == chatbob_normalized
    normalized_dat_results[str(relative_path)] = identical

    status = "IDENTICAL" if identical else "DIFFERENT"
    print(f"{status:10} {relative_path}")

NORMALIZED DAT COMPARISON
IDENTICAL  CURATE/CURATE_data.dat
IDENTICAL  GENERATE/GENERATE_data.dat
IDENTICAL  PREDICT/PREDICT_data.dat
IDENTICAL  VERIFY/VERIFY_data.dat


In [12]:
# Final validation summary

dat_pass = all(normalized_dat_results.values())

csv_failures = []

for relative_path, result in csv_comparison_results.items():
    if relative_path == "CURATE/CURATE_options.csv":
        continue

    if not (
        result["same_shape"]
        and result["same_columns"]
        and result["numeric_differences"] == 0
        and result["text_differences"] == 0
    ):
        csv_failures.append(relative_path)

print("ROBERT OUTPUT VALIDATION SUMMARY")
print("================================")

print(f"Normalized DAT comparison: {'PASS' if dat_pass else 'FAIL'}")
print(f"Scientific CSV comparison: {'PASS' if not csv_failures else 'FAIL'}")

print(
    "Expected metadata difference: "
    "CURATE/CURATE_options.csv records a different input CSV path."
)

if not dat_pass or csv_failures:
    print("\nConclusion: Differences require investigation.")

    if not dat_pass:
        print("  One or more normalized DAT comparisons failed.")

    if csv_failures:
        print("  CSV files requiring investigation:")
        for path in csv_failures:
            print(f"    {path}")
else:
    print(
        "\nConclusion: The ChatBob-modified ROBERT 2.1.2 preserves "
        "the standard ROBERT scientific outputs for this test case."
    )

ROBERT OUTPUT VALIDATION SUMMARY
Normalized DAT comparison: PASS
Scientific CSV comparison: PASS
Expected metadata difference: CURATE/CURATE_options.csv records a different input CSV path.

Conclusion: The ChatBob-modified ROBERT 2.1.2 preserves the standard ROBERT scientific outputs for this test case.
